In [7]:
import pymongo
import sqlite3
import pandas as pd

client = pymongo.MongoClient()
db = client.SAE

conn = sqlite3.connect("ClassicModel.sqlite")

In [8]:
employees = pd.read_sql_query("SELECT * FROM Employees;", conn)
offices = pd.read_sql_query("SELECT * FROM Offices;", conn)

# Liste des employés par office
employees_by_office = [
    employees.query("officeCode == @code")
             .drop(columns=["officeCode"])
             .to_dict(orient="records")
    for code in offices.officeCode
]

offices_mongo = offices.assign(employees=employees_by_office)

db.offices.insert_many(offices_mongo.to_dict(orient="records"))

InsertManyResult([ObjectId('693c0cddd3467b0721d85e05'), ObjectId('693c0cddd3467b0721d85e06'), ObjectId('693c0cddd3467b0721d85e07'), ObjectId('693c0cddd3467b0721d85e08'), ObjectId('693c0cddd3467b0721d85e09'), ObjectId('693c0cddd3467b0721d85e0a'), ObjectId('693c0cddd3467b0721d85e0b')], acknowledged=True)

In [ ]:
orders = pd.read_sql_query("SELECT * FROM Orders;", conn)
customers = pd.read_sql_query("SELECT * FROM Customers;", conn)

orders_by_customer = [
    orders.query("customerNumber == @cid")
          .drop(columns=["customerNumber"])
          .to_dict(orient="records")
    for cid in customers.customerNumber
]

customers_mongo = customers.assign(orders=orders_by_customer)

db.customers.insert_many(customers_mongo.to_dict(orient="records"))

InsertManyResult([ObjectId('693c0cddd3467b0721d85e0c'), ObjectId('693c0cddd3467b0721d85e0d'), ObjectId('693c0cddd3467b0721d85e0e'), ObjectId('693c0cddd3467b0721d85e0f'), ObjectId('693c0cddd3467b0721d85e10'), ObjectId('693c0cddd3467b0721d85e11'), ObjectId('693c0cddd3467b0721d85e12'), ObjectId('693c0cddd3467b0721d85e13'), ObjectId('693c0cddd3467b0721d85e14'), ObjectId('693c0cddd3467b0721d85e15'), ObjectId('693c0cddd3467b0721d85e16'), ObjectId('693c0cddd3467b0721d85e17'), ObjectId('693c0cddd3467b0721d85e18'), ObjectId('693c0cddd3467b0721d85e19'), ObjectId('693c0cddd3467b0721d85e1a'), ObjectId('693c0cddd3467b0721d85e1b'), ObjectId('693c0cddd3467b0721d85e1c'), ObjectId('693c0cddd3467b0721d85e1d'), ObjectId('693c0cddd3467b0721d85e1e'), ObjectId('693c0cddd3467b0721d85e1f'), ObjectId('693c0cddd3467b0721d85e20'), ObjectId('693c0cddd3467b0721d85e21'), ObjectId('693c0cddd3467b0721d85e22'), ObjectId('693c0cddd3467b0721d85e23'), ObjectId('693c0cddd3467b0721d85e24'), ObjectId('693c0cddd3467b0721d85e

In [ ]:
order_product = pd.read_sql_query("""
    SELECT O.*, 
           P.productName, 
           P.productLine, 
           P.productScale,
           P.productVendor, 
           P.productDescription, 
           P.quantityInStock, 
           P.buyPrice, 
           P.MSRP
    FROM OrderDetails O
    LEFT JOIN Products P ON O.productCode = P.productCode;
""", conn)

orders_full = []

for oid in orders.orderNumber:
    details = order_product.query("orderNumber == @oid").to_dict(orient="records")
    
    order_mongo = orders.query("orderNumber == @oid").iloc[0].to_dict()
    order_mongo["details"] = details
    
    orders_full.append(order_mongo)

db.orders.insert_many(orders_full)

InsertManyResult([ObjectId('693c0cddd3467b0721d85e86'), ObjectId('693c0cddd3467b0721d85e87'), ObjectId('693c0cddd3467b0721d85e88'), ObjectId('693c0cddd3467b0721d85e89'), ObjectId('693c0cddd3467b0721d85e8a'), ObjectId('693c0cddd3467b0721d85e8b'), ObjectId('693c0cddd3467b0721d85e8c'), ObjectId('693c0cddd3467b0721d85e8d'), ObjectId('693c0cddd3467b0721d85e8e'), ObjectId('693c0cddd3467b0721d85e8f'), ObjectId('693c0cddd3467b0721d85e90'), ObjectId('693c0cddd3467b0721d85e91'), ObjectId('693c0cddd3467b0721d85e92'), ObjectId('693c0cddd3467b0721d85e93'), ObjectId('693c0cddd3467b0721d85e94'), ObjectId('693c0cddd3467b0721d85e95'), ObjectId('693c0cddd3467b0721d85e96'), ObjectId('693c0cddd3467b0721d85e97'), ObjectId('693c0cddd3467b0721d85e98'), ObjectId('693c0cddd3467b0721d85e99'), ObjectId('693c0cddd3467b0721d85e9a'), ObjectId('693c0cddd3467b0721d85e9b'), ObjectId('693c0cddd3467b0721d85e9c'), ObjectId('693c0cddd3467b0721d85e9d'), ObjectId('693c0cddd3467b0721d85e9e'), ObjectId('693c0cddd3467b0721d85e

In [11]:
payments = pd.read_sql_query("SELECT * FROM Payments;", conn)
db.payments.insert_many(payments.to_dict(orient="records"))

conn.close()